[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/38.%20The%20Automation%20Investment%20Analysis%20Problem/P38-Tier-4_executed.ipynb)

---


# 38. The Automation Investment Analysis Problem
## Tier 4: Deep Q-Network Reinforcement Learning

### Goal

Use a Deep Q-Network (DQN) to learn a sequential investment policy. At each decision, the agent can select an affordable project that has not already been selected, or stop investing. A simulated market condition is sampled for each episode. The agent receives the portfolio reward when the episode ends and learns from stored state transitions.

The notebook implements the DQN solution sequence: define an environment and state representation, approximate action values with a neural network, collect experience using an epsilon-greedy policy, train from replayed transitions, and evaluate the learned policy.

### Inputs and outputs

The inputs are the five project descriptions, a budget of $50 million, three possible market conditions, and the episode and agent settings in the code. The outputs are training metrics, test rewards and selected portfolios, policy summaries, and visualizations. Results are generated when the notebook is run; no particular outcome is assumed in advance.

### Scope and modeling assumptions

- Each episode has one market condition, sampled from `env.market_conditions`. That condition remains fixed during the episode; the model does not simulate market changes within an episode.
- Project costs, base benefits, risk factors, strategic values, and technology maturities are fixed inputs. The state includes these project features, while the reward uses base benefit, risk, strategic value, market condition, budget use, and specified synergies.
- Investment decisions are sequential, but selecting a project does not earn an immediate reward. The portfolio receives its reward when the agent stops or reaches the investment limit.
- The reward is a constructed model score, not a discounted cash-flow calculation or a validated financial return. Project costs are expressed in millions of dollars; reward units are illustrative.
- This small simulated example does not use historical data, model project dependencies beyond the stated synergies, or represent uncertainty in project outcomes. DQN performance depends on training and is not guaranteed to be globally optimal.

### Learning objectives

By following the notebook, you will connect an investment problem to a Markov decision process, interpret a Bellman target, understand the roles of neural-network value approximation and experience replay, and assess the limits of an empirical policy evaluation.

## 1. Mathematical model and DQN learning rule

### From portfolio construction to a sequential decision problem

A state records the market condition, remaining budget, the number of decisions already made, which projects have been selected, and the fixed features of every project. The agent's action is either to select one feasible project or to stop. The set of feasible project actions changes as the portfolio grows and the remaining budget falls.

Formally, let \(s_t\) be the state at decision \(t\), \(a_t\) a valid action, \(r_t\) the reward, and \(s_{t+1}\) the next state. Together, these quantities define a Markov decision process (MDP): the next state and reward depend on the current state and action, rather than on the full history once the state is known. Here, the environment is simulated by the notebook's state-construction, action-validity, and reward functions.

### Portfolio reward

For a selected portfolio \(S\), market condition \(m\), and project \(i\), define the risk-adjusted project benefit, including its strategic-value bonus, as

$$
b_i(m) = b_i^{\mathrm{base}}\,\mu_m\,(1 - 0.3\,\rho_i) + 2\,v_i.
$$

The portfolio score is the sum of these benefits, multiplied by the applicable synergy and budget-utilization factors:

$$
R(S,m) = \left(\sum_{i\in S} b_i(m)\right)
         \left(1 + 0.15\,\mathbf{1}_{\{0,1\}\subseteq S}\right)
         \left(1 + 0.10\,\mathbf{1}_{\{3,4\}\subseteq S}\right)
         \left(1 + 0.10\,\mathbf{1}_{0.7\leq C(S)/B\leq 1}\right),
$$

when \(S\) is nonempty and within budget. Here, \(b_i^{\mathrm{base}}\) is project \(i\)'s base benefit, \(μ_m\) is the market multiplier, \(ρ_i\) is its risk factor, \(v_i\) is its strategic value, \(C(S)\) is total project cost, and \(B\) is the budget. The indicator \({1}_A\) equals 1 when condition \(A\) holds and 0 otherwise. The code applies the crane-and-yard synergy to projects 0 and 1, the maintenance-and-vehicles synergy to projects 3 and 4, and the budget-utilization bonus when utilization is between 70% and 100%, inclusive. An empty portfolio receives a penalty; an over-budget portfolio has an additional penalty, although feasible-action filtering prevents the agent from constructing one.

### Action values and the Bellman target

The action-value function \(Q(s,a)\) estimates the expected discounted return from taking action \(a\) in state \(s\), then following the policy. DQN approximates this function with a neural network whose inputs are state vectors and whose outputs are one value per action. For a sampled transition, the learning target is

$$
y_t = r_t + \gamma(1-d_t)\max_{a'\in A(s_{t+1})} Q_{\mathrm{target}}(s_{t+1},a'),
$$

where \(γ\) is the discount factor, \(d_t=1\) for a terminal transition and 0 otherwise, and \(A(s_{t+1})\) is the set of valid next actions. A terminal transition has no future-value term. Restricting the maximum to valid actions matters because an unaffordable or already-selected project is not a possible next decision. The agent updates the value for the action actually taken toward this target; values for other actions are left unchanged in that training example.

The target network supplies the next-state values and is periodically copied from the trainable Q-network. Experience replay stores transitions and samples a random minibatch, reducing the dependence of each update on the most recent transition alone. Epsilon-greedy action selection explores with probability ε and otherwise selects the valid action with the largest estimated Q-value; ε decays during training.

### Termination and computational scale

An episode ends when the agent chooses the stop action, reaches `env.max_investments`, or has no feasible project left. The portfolio reward is assigned at that terminal point. If there are \(n\) projects, a portfolio can have up to \(2^n\) subsets before budget and decision-limit restrictions. DQN does not enumerate all subsets: its work is determined by the number of training transitions, minibatch size, and neural-network dimensions. This does not guarantee that training will find the best portfolio.

In [ ]:
from collections import Counter, deque, namedtuple
from dataclasses import dataclass
import math
import random
from time import time
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported for DQN investment learning.")

## 2. Neural network and environment structures

The neural network has one hidden ReLU layer and a linear output layer. Its output contains one estimated action value per possible action. Backpropagation uses the difference between the network's current predictions and the DQN targets to update the weights by gradient descent.

The environment data structure groups the fixed project descriptions with the budget, market-condition labels, and maximum number of investment decisions. Keeping these inputs explicit makes it clear which information defines an episode.

In [ ]:
class SimpleNeuralNetwork:
    """One-hidden-layer network used to approximate action values."""

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        learning_rate: float = 0.001,
    ):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.learning_rate = learning_rate

        self.W1 = np.random.randn(input_size, hidden_size) * 0.1
        self.b1 = np.zeros(hidden_size)
        self.W2 = np.random.randn(hidden_size, output_size) * 0.1
        self.b2 = np.zeros(output_size)

    @staticmethod
    def relu(x: np.ndarray) -> np.ndarray:
        return np.maximum(0, x)

    @staticmethod
    def relu_derivative(x: np.ndarray) -> np.ndarray:
        return (x > 0).astype(float)

    def forward(self, x: np.ndarray) -> np.ndarray:
        """Map a batch of state vectors to action-value estimates."""
        self.z1 = np.dot(x, self.W1) + self.b1
        self.a1 = self.relu(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        return self.z2

    def backward(
        self,
        x: np.ndarray,
        target: np.ndarray,
        predicted: np.ndarray,
    ) -> None:
        """Update weights using the batch prediction error."""
        batch_size = x.shape[0]

        dz2 = (predicted - target) / batch_size
        dW2 = np.dot(self.a1.T, dz2)
        db2 = np.sum(dz2, axis=0)

        dz1 = np.dot(dz2, self.W2.T) * self.relu_derivative(self.z1)
        dW1 = np.dot(x.T, dz1)
        db1 = np.sum(dz1, axis=0)

        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1

    def predict(self, x: np.ndarray) -> np.ndarray:
        """Return the network's current action-value estimates."""
        return self.forward(x)


@dataclass
class InvestmentEnvironment:
    """Fixed project data and limits for the investment simulation."""

    projects: List[Dict]
    budget: float
    market_conditions: List[str]
    max_investments: int


Experience = namedtuple(
    "Experience",
    ["state", "action", "reward", "next_state", "done", "next_valid_actions"],
)

print("Neural network and environment structures defined.")

## 3. Project data and investment environment

The project records provide the features used by the state representation and reward calculation. Project IDs are used as action indices, so they must be consecutive integers starting at zero and correspond to the projects' positions in the list. The budget and decision limit constrain which portfolios can be built.

In [ ]:
projects_data = [
    {
        "id": 0,
        "name": "Autonomous Crane System",
        "cost": 20.0,
        "base_benefit": 5.0,
        "risk_factor": 0.25,
        "strategic_value": 0.90,
        "tech_maturity": 0.80,
    },
    {
        "id": 1,
        "name": "AI Yard Management",
        "cost": 15.0,
        "base_benefit": 4.0,
        "risk_factor": 0.15,
        "strategic_value": 0.85,
        "tech_maturity": 0.95,
    },
    {
        "id": 2,
        "name": "Digital Twin Platform",
        "cost": 18.0,
        "base_benefit": 4.0,
        "risk_factor": 0.35,
        "strategic_value": 0.95,
        "tech_maturity": 0.70,
    },
    {
        "id": 3,
        "name": "Predictive Maintenance AI",
        "cost": 12.0,
        "base_benefit": 3.5,
        "risk_factor": 0.10,
        "strategic_value": 0.70,
        "tech_maturity": 0.90,
    },
    {
        "id": 4,
        "name": "Autonomous Vehicles",
        "cost": 25.0,
        "base_benefit": 6.0,
        "risk_factor": 0.40,
        "strategic_value": 0.88,
        "tech_maturity": 0.75,
    },
]

env = InvestmentEnvironment(
    projects=projects_data,
    budget=50.0,
    market_conditions=["High", "Base", "Low"],
    max_investments=5,
)

assert [project["id"] for project in env.projects] == list(range(len(env.projects)))
assert env.budget > 0
assert env.max_investments > 0

print(f"Created an investment environment with {len(env.projects)} projects.")
print(f"Budget: ${env.budget:.1f}M; maximum project selections: {env.max_investments}.")

## 4. DQN agent

The state vector combines a one-hot market-condition encoding, normalized budget and decision count, one selected/not-selected indicator per project, and five project features per project. Normalization places quantities with different scales into more comparable numerical ranges; it does not change the underlying project data.

The action space has one action for each project and one additional stop action. At decision time, the agent chooses only from the valid-action list. In replay, the Bellman target also maximizes only over valid next actions. This keeps both policy selection and learning targets consistent with the budget and selection rules.

In [ ]:
class DQNAgent:
    """DQN agent with a target network and experience replay."""

    def __init__(self, state_size: int, action_size: int, hidden_size: int = 64):
        self.state_size = state_size
        self.action_size = action_size
        self.hidden_size = hidden_size

        self.q_network = SimpleNeuralNetwork(
            state_size, hidden_size, action_size, learning_rate=0.001
        )
        self.target_network = SimpleNeuralNetwork(
            state_size, hidden_size, action_size, learning_rate=0.001
        )
        self.update_target_network()

        self.memory = deque(maxlen=2000)

        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.batch_size = 32
        self.update_target_every = 10

        self.training_loss = []
        self.epsilon_history = []

    def update_target_network(self) -> None:
        """Synchronize target values with the trainable network."""
        self.target_network.W1 = self.q_network.W1.copy()
        self.target_network.b1 = self.q_network.b1.copy()
        self.target_network.W2 = self.q_network.W2.copy()
        self.target_network.b2 = self.q_network.b2.copy()

    def get_state(
        self,
        env: InvestmentEnvironment,
        market_condition: str,
        selected_projects: List[int],
        remaining_budget: float,
        step: int,
    ) -> np.ndarray:
        """Encode the current episode information as a numeric vector."""
        market_encoding = {
            "High": [1, 0, 0],
            "Base": [0, 1, 0],
            "Low": [0, 0, 1],
        }
        if market_condition not in market_encoding:
            raise ValueError(f"Unknown market condition: {market_condition}")

        state = list(market_encoding[market_condition])
        state.append(remaining_budget / env.budget)
        state.append(step / env.max_investments)

        for project in env.projects:
            state.append(1.0 if project["id"] in selected_projects else 0.0)

        for project in env.projects:
            state.extend(
                [
                    project["cost"] / 50.0,
                    project["base_benefit"] / 10.0,
                    project["risk_factor"],
                    project["strategic_value"],
                    project["tech_maturity"],
                ]
            )

        return np.asarray(state, dtype=float)

    def act(self, state: np.ndarray, valid_actions: List[int]) -> int:
        """Choose a valid action using an epsilon-greedy policy."""
        if not valid_actions:
            raise ValueError("At least one valid action is required.")

        if np.random.random() <= self.epsilon:
            return random.choice(valid_actions)

        q_values = self.q_network.predict(state.reshape(1, -1))[0]
        return max(valid_actions, key=lambda action: q_values[action])

    def remember(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        next_valid_actions: Optional[List[int]] = None,
    ) -> None:
        """Store a transition and its feasible next actions for replay."""
        valid_actions = [] if next_valid_actions is None else list(next_valid_actions)
        self.memory.append(
            Experience(state, action, reward, next_state, done, valid_actions)
        )

    def replay(self, batch_size: Optional[int] = None) -> None:
        """Train on a randomly sampled batch of stored transitions."""
        if batch_size is None:
            batch_size = self.batch_size
        if len(self.memory) < batch_size:
            return

        batch = random.sample(self.memory, batch_size)
        states = np.asarray([experience.state for experience in batch])
        actions = np.asarray([experience.action for experience in batch], dtype=int)
        rewards = np.asarray([experience.reward for experience in batch], dtype=float)
        next_states = np.asarray([experience.next_state for experience in batch])
        dones = np.asarray([experience.done for experience in batch], dtype=bool)

        current_q_values = self.q_network.predict(states)
        next_q_values = self.target_network.predict(next_states)
        target_q_values = current_q_values.copy()

        for index, experience in enumerate(batch):
            if experience.done:
                target_q_values[index, actions[index]] = rewards[index]
            elif experience.next_valid_actions:
                best_next_value = max(
                    next_q_values[index, action]
                    for action in experience.next_valid_actions
                )
                target_q_values[index, actions[index]] = (
                    rewards[index] + self.gamma * best_next_value
                )
            else:
                raise ValueError(
                    "A nonterminal transition must have a valid next action."
                )

        self.q_network.backward(states, target_q_values, current_q_values)
        loss = np.mean(np.square(target_q_values - current_q_values))
        self.training_loss.append(float(loss))

    def update_epsilon(self) -> None:
        """Reduce exploration while retaining a minimum exploration rate."""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        self.epsilon_history.append(self.epsilon)

print("DQN agent implementation complete.")

## 5. Reward and feasible actions

`calculate_reward` implements the portfolio score defined above. `get_valid_actions` returns the stop action plus every project that is both unselected and affordable with the remaining budget. The stop action is always available, so a valid decision is possible even when no further project can be selected.

The reward is delayed until the portfolio is complete. This makes terminal handling important: the final reward must be attached to the transition that ends the episode, whether the agent explicitly stops or reaches a limit.

In [ ]:
def calculate_reward(
    env: InvestmentEnvironment,
    selected_projects: List[int],
    market_condition: str,
    remaining_budget: float,
) -> float:
    """Score a completed portfolio using the model's specified reward terms."""
    if not selected_projects:
        return -10.0

    total_cost = sum(env.projects[index]["cost"] for index in selected_projects)
    if total_cost > env.budget:
        return -(total_cost - env.budget) * 5.0

    market_multipliers = {"High": 1.5, "Base": 1.0, "Low": 0.6}
    if market_condition not in market_multipliers:
        raise ValueError(f"Unknown market condition: {market_condition}")

    multiplier = market_multipliers[market_condition]
    total_benefit = 0.0

    for project_id in selected_projects:
        project = env.projects[project_id]
        benefit = project["base_benefit"] * multiplier
        benefit *= 1 - project["risk_factor"] * 0.3
        benefit += project["strategic_value"] * 2.0
        total_benefit += benefit

    if 0 in selected_projects and 1 in selected_projects:
        total_benefit *= 1.15

    if 3 in selected_projects and 4 in selected_projects:
        total_benefit *= 1.10

    budget_utilization = total_cost / env.budget
    if 0.7 <= budget_utilization <= 1.0:
        total_benefit *= 1.1

    return total_benefit


def get_valid_actions(
    env: InvestmentEnvironment,
    selected_projects: List[int],
    remaining_budget: float,
) -> List[int]:
    """List the affordable unselected projects and the always-available stop action."""
    stop_action = len(env.projects)
    valid_actions = [stop_action]

    for index, project in enumerate(env.projects):
        if index not in selected_projects and project["cost"] <= remaining_budget:
            valid_actions.append(index)

    return valid_actions

print("Reward and feasible-action functions defined.")

## 6. Train the DQN agent

Each episode samples one market condition and starts with an empty portfolio and the full budget. At each step, the agent chooses among affordable unselected projects and the stop action. A project-selection transition has zero reward unless it ends the episode; a terminal transition receives the completed portfolio score. This reward assignment makes the return from a sequence of selections correspond to the final portfolio reward.

After each episode, replay performs a minibatch update when enough transitions have been collected. Epsilon decays to shift the policy gradually from exploration toward value-based choices, and the target network is synchronized periodically. The state and action dimensions are calculated from the environment so the network matches the project set.

In [ ]:
def train_dqn_agent(
    env: InvestmentEnvironment,
    num_episodes: int = 1000,
) -> Tuple[DQNAgent, List[float], List[int]]:
    """Train a DQN agent and return it with episode rewards and decision counts."""
    state_size = 3 + 1 + 1 + len(env.projects) + len(env.projects) * 5
    action_size = len(env.projects) + 1
    stop_action = len(env.projects)

    agent = DQNAgent(state_size, action_size, hidden_size=64)
    episode_rewards = []
    episode_lengths = []

    print(f"Training the DQN agent for {num_episodes} episodes...")
    print(f"State size: {state_size}; action size: {action_size}.")

    for episode in range(num_episodes):
        market_condition = random.choice(env.market_conditions)
        selected_projects = []
        remaining_budget = env.budget
        total_reward = 0.0
        step = 0

        while step < env.max_investments:
            state = agent.get_state(
                env, market_condition, selected_projects, remaining_budget, step
            )
            valid_actions = get_valid_actions(
                env, selected_projects, remaining_budget
            )
            action = agent.act(state, valid_actions)

            if action == stop_action:
                reward = calculate_reward(
                    env, selected_projects, market_condition, remaining_budget
                )
                agent.remember(state, action, reward, state.copy(), True, [])
                total_reward += reward
                step += 1
                break

            selected_projects.append(action)
            remaining_budget -= env.projects[action]["cost"]
            step += 1

            next_state = agent.get_state(
                env, market_condition, selected_projects, remaining_budget, step
            )
            next_valid_actions = get_valid_actions(
                env, selected_projects, remaining_budget
            )
            has_feasible_project = any(
                next_action != stop_action for next_action in next_valid_actions
            )
            done = step >= env.max_investments or not has_feasible_project

            reward = (
                calculate_reward(
                    env, selected_projects, market_condition, remaining_budget
                )
                if done
                else 0.0
            )
            agent.remember(
                state,
                action,
                reward,
                next_state,
                done,
                [] if done else next_valid_actions,
            )
            total_reward += reward

            if done:
                break

        episode_rewards.append(total_reward)
        episode_lengths.append(step)

        if len(agent.memory) >= agent.batch_size:
            agent.replay()

        agent.update_epsilon()

        if (episode + 1) % agent.update_target_every == 0:
            agent.update_target_network()

        if episode % 100 == 0 or episode == num_episodes - 1:
            average_reward = np.mean(episode_rewards[-100:])
            print(
                f"Episode {episode + 1:4d}: "
                f"Average reward = {average_reward:.2f}, "
                f"epsilon = {agent.epsilon:.3f}"
            )

    return agent, episode_rewards, episode_lengths

print("Training function defined.")

## 7. Train and inspect training metrics

The training cell runs the configured number of episodes and reports summary statistics calculated from the run. Early-versus-late average reward is a descriptive comparison, not proof of convergence: episodes use sampled market conditions, and the training process is stochastic. The network dimensions and final exploration rate help document the model that was trained.

In [ ]:
start_time = time()
agent, training_rewards, training_lengths = train_dqn_agent(
    env, num_episodes=1000
)
training_time = time() - start_time

print("\n" + "=" * 72)
print("DQN TRAINING RESULTS")
print("=" * 72)
print(f"\nTraining time: {training_time:.2f} seconds")
print(f"Episodes completed: {len(training_rewards)}")
print(f"Final epsilon: {agent.epsilon:.4f}")
print(f"Replay memory size: {len(agent.memory)} experiences")

if training_rewards:
    early_reward = float(np.mean(training_rewards[:100]))
    late_reward = float(np.mean(training_rewards[-100:]))
    reward_change = late_reward - early_reward

    print("\nLearning progress (descriptive):")
    print(f"First 100 episodes, average reward: {early_reward:.2f}")
    print(f"Last 100 episodes, average reward: {late_reward:.2f}")
    print(f"Change in average reward: {reward_change:.2f}")
    print(f"Average decisions per episode: {np.mean(training_lengths):.2f}")
    print(f"Maximum episode reward: {max(training_rewards):.2f}")
    print(f"Minimum episode reward: {min(training_rewards):.2f}")

print("\nNeural-network architecture:")
print(f"Input layer: {agent.q_network.input_size} units")
print(f"Hidden layer: {agent.q_network.hidden_size} units")
print(f"Output layer: {agent.q_network.output_size} units")
print(f"Learning rate: {agent.q_network.learning_rate}")

## 8. Test the learned policy

Testing sets epsilon to zero so the agent chooses the highest-valued valid action without exploratory random choices. Market conditions are still sampled for each test episode, so test episodes can differ in their conditions. The returned condition labels allow rewards and portfolios to be grouped by the condition that actually generated them.

In [ ]:
def test_agent(
    agent: DQNAgent,
    env: InvestmentEnvironment,
    num_test_episodes: int = 100,
) -> Tuple[List[float], List[List[int]], List[str]]:
    """Evaluate the greedy policy and return rewards, portfolios, and markets."""
    test_rewards = []
    test_selections = []
    test_market_conditions = []
    stop_action = len(env.projects)
    original_epsilon = agent.epsilon

    print(f"\nTesting the trained agent for {num_test_episodes} episodes...")
    agent.epsilon = 0.0

    try:
        for episode in range(num_test_episodes):
            market_condition = random.choice(env.market_conditions)
            selected_projects = []
            remaining_budget = env.budget
            step = 0

            while step < env.max_investments:
                state = agent.get_state(
                    env, market_condition, selected_projects, remaining_budget, step
                )
                valid_actions = get_valid_actions(
                    env, selected_projects, remaining_budget
                )
                action = agent.act(state, valid_actions)

                if action == stop_action:
                    break

                selected_projects.append(action)
                remaining_budget -= env.projects[action]["cost"]
                step += 1

                next_valid_actions = get_valid_actions(
                    env, selected_projects, remaining_budget
                )
                has_feasible_project = any(
                    next_action != stop_action for next_action in next_valid_actions
                )
                if step >= env.max_investments or not has_feasible_project:
                    break

            reward = calculate_reward(
                env, selected_projects, market_condition, remaining_budget
            )
            test_rewards.append(reward)
            test_selections.append(selected_projects.copy())
            test_market_conditions.append(market_condition)

            if episode % 20 == 0:
                print(f"Test episode {episode + 1:3d}: reward = {reward:.2f}")
    finally:
        agent.epsilon = original_epsilon

    return test_rewards, test_selections, test_market_conditions


test_rewards, test_selections, test_market_conditions = test_agent(
    agent, env, num_test_episodes=100
)

assert len(test_rewards) == len(test_selections) == len(test_market_conditions)

print("\nTest results:")
print(f"Average test reward: {np.mean(test_rewards):.2f}")
print(f"Test reward standard deviation: {np.std(test_rewards):.2f}")
print(f"Best test reward: {max(test_rewards):.2f}")
print(f"Worst test reward: {min(test_rewards):.2f}")

selection_counts = Counter(
    tuple(sorted(selection)) for selection in test_selections
)
print("\nMost common investment strategies:")
for rank, (selection, count) in enumerate(selection_counts.most_common(5), start=1):
    project_names = [env.projects[index]["name"] for index in selection]
    total_cost = sum(env.projects[index]["cost"] for index in selection)
    frequency = 100 * count / len(test_selections)
    print(
        f"{rank}. {project_names} "
        f"(cost: ${total_cost:.1f}M; frequency: {frequency:.1f}%)"
    )

## 9. Visualize training and test performance

The panels show episode rewards alongside a moving average, the exploration-rate schedule, replay training loss, and the distribution of test rewards. Moving averages make broad patterns easier to inspect but do not remove randomness or establish statistical significance. The test histogram describes this run's sampled test episodes; it is not a confidence interval or a comparison with another method.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
    2, 2, figsize=(16, 12)
)

window_size = 50
smoothed_rewards = [
    np.mean(training_rewards[max(0, index - window_size + 1): index + 1])
    for index in range(len(training_rewards))
]
episodes = np.arange(1, len(training_rewards) + 1)

ax1.plot(episodes, smoothed_rewards, color="blue", linewidth=2, label="Moving average")
ax1.plot(
    episodes,
    training_rewards,
    color="lightblue",
    alpha=0.35,
    linewidth=1,
    label="Episode reward",
)
ax1.set_xlabel("Episode")
ax1.set_ylabel("Reward (model units)")
ax1.set_title("DQN Training Rewards")
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(
    np.arange(1, len(agent.epsilon_history) + 1),
    agent.epsilon_history,
    color="red",
    linewidth=2,
)
ax2.set_xlabel("Episode")
ax2.set_ylabel("Epsilon (exploration probability)")
ax2.set_title("Exploration-Rate Decay")
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3)

if agent.training_loss:
    loss_window = 100
    smoothed_loss = [
        np.mean(agent.training_loss[max(0, index - loss_window + 1): index + 1])
        for index in range(len(agent.training_loss))
    ]
    ax3.plot(
        np.arange(1, len(smoothed_loss) + 1),
        smoothed_loss,
        color="green",
        linewidth=2,
    )
ax3.set_xlabel("Replay training update")
ax3.set_ylabel("Mean squared error")
ax3.set_title("DQN Training Loss")
ax3.grid(True, alpha=0.3)

ax4.hist(test_rewards, bins=20, alpha=0.7, color="purple", edgecolor="black")
ax4.axvline(
    np.mean(test_rewards),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {np.mean(test_rewards):.2f}",
)
ax4.axvline(
    np.median(test_rewards),
    color="orange",
    linestyle="--",
    linewidth=2,
    label=f"Median: {np.median(test_rewards):.2f}",
)
ax4.set_xlabel("Test reward (model units)")
ax4.set_ylabel("Number of episodes")
ax4.set_title("Test-Reward Distribution")
ax4.legend()
ax4.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()
print("DQN training and evaluation visualizations complete.")

## 10. Policy and action-value analysis

The market-condition summary uses the condition recorded during each test episode, rather than assigning rewards to conditions after testing. The action-value inspection reports the network's estimates at an initial sample state. At that state all projects are initially affordable, but Q-values are estimates, not rewards or probabilities; a large value does not by itself establish that an action is optimal.

Policy consistency is reported as the frequency of the most common selected portfolio. A consistent policy can still perform poorly, and a changing policy is not necessarily better. These summaries should be interpreted together with rewards, feasible actions, and the limitations of this simulated environment.

In [ ]:
print("\n" + "=" * 72)
print("POLICY ANALYSIS AND DECISION PATTERNS")
print("=" * 72)

print("\nPerformance by sampled market condition:")
for condition in env.market_conditions:
    condition_indices = [
        index
        for index, observed_condition in enumerate(test_market_conditions)
        if observed_condition == condition
    ]
    if not condition_indices:
        print(f"- {condition} market: no test episodes were sampled.")
        continue

    condition_rewards = [test_rewards[index] for index in condition_indices]
    condition_selections = [test_selections[index] for index in condition_indices]
    condition_strategies = Counter(
        tuple(sorted(selection)) for selection in condition_selections
    )
    common_strategy, common_count = condition_strategies.most_common(1)[0]
    common_names = [env.projects[index]["name"] for index in common_strategy]

    print(
        f"- {condition} market: average reward = "
        f"{np.mean(condition_rewards):.2f} "
        f"across {len(condition_indices)} episodes"
    )
    print(
        f"  Most common portfolio: {common_names} "
        f"({common_count} of {len(condition_indices)} episodes)"
    )

sample_market = "Base"
sample_state = agent.get_state(env, sample_market, [], env.budget, 0)
sample_q_values = agent.q_network.predict(sample_state.reshape(1, -1))[0]
sample_valid_actions = get_valid_actions(env, [], env.budget)

print("\nAction-value estimates for an initial sample state:")
print(f"Market condition: {sample_market}")
print(f"Remaining budget: ${env.budget:.1f}M")
for action, q_value in enumerate(sample_q_values):
    if action < len(env.projects):
        action_name = f"Select {env.projects[action]['name']}"
    else:
        action_name = "Stop investing"
    validity = "valid" if action in sample_valid_actions else "invalid"
    print(f"{action_name}: Q = {q_value:.3f} ({validity} action)")

if len(training_rewards) >= 100:
    first_100_average = float(np.mean(training_rewards[:100]))
    last_100_average = float(np.mean(training_rewards[-100:]))
    change = last_100_average - first_100_average

    if math.isclose(change, 0.0, rel_tol=1e-9, abs_tol=1e-9):
        assessment = "approximately unchanged"
    elif change > 0:
        assessment = "higher in the final 100 episodes"
    else:
        assessment = "lower in the final 100 episodes"

    print("\nDescriptive learning assessment:")
    print(f"First 100-episode average reward: {first_100_average:.2f}")
    print(f"Last 100-episode average reward: {last_100_average:.2f}")
    print(f"Change: {change:.2f} ({assessment})")

if test_selections:
    unique_strategies = len(
        {tuple(sorted(selection)) for selection in test_selections}
    )
    most_common_count = selection_counts.most_common(1)[0][1]
    consistency = 100 * most_common_count / len(test_selections)

    print("\nPolicy consistency:")
    print(f"Distinct portfolios observed: {unique_strategies}")
    print(f"Test episodes: {len(test_selections)}")
    print(f"Most common portfolio frequency: {consistency:.1f}%")

print("\n" + "=" * 72)
print("DQN ANALYSIS COMPLETE")
print("=" * 72)